# Aegis Support Agent Demo

Aegis is a trust-gated customer support agent for B2B SaaS teams.

It demonstrates:

- Multi-agent workflow
- Policy lookup tool use
- MCP-style tool interface
- A2A-style structured communication
- Judge agent evaluation
- Trust scoring
- Human-in-the-loop escalation


## 1. Project Setup

This notebook assumes it is run from the project root:

aegis-support-agent/


If running from the `notebooks/` folder, move one directory up first.


In [ ]:
import os
from pathlib import Path

# If running inside notebooks/, move to project root
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Current directory:", Path.cwd())
print("Project files:", [p.name for p in Path.cwd().iterdir() if p.name in ["src", "data", "policies", "README.md"]])


## 2. Load Sample Tickets

The MVP uses a small synthetic support-ticket dataset.


In [ ]:
import pandas as pd

df = pd.read_csv("data/sample_tickets.csv")
df


## 3. Run Aegis on One Ticket

The pipeline processes one customer ticket through:

Triage Agent → Policy Lookup Tool → Draft Response Agent → Judge Agent → Trust Gate



In [ ]:
from src.main import process_ticket

ticket = "I was charged twice this month. Please refund me now."

result = process_ticket("demo-001", ticket)
result.model_dump()


## 4. Inspect the Draft Response

The draft is not automatically trusted. It is evaluated by the judge agent first.


In [ ]:
print(result.draft_response)


## 5. Inspect the Trust Gate Decision

The trust gate decides whether the response can be suggested directly, needs human review, or must be escalated.


In [ ]:
print("Category:", result.category)
print("Risk level:", result.risk_level)
print("Trust score:", result.trust_score)
print("Decision:", result.decision)
print("Reason:", result.reason)

if result.escalation_note:
    print()
    print("Escalation note:")
    print(result.escalation_note)


## 6. Run All Sample Tickets

Now we run the full sample dataset.


In [ ]:
results = []

for _, row in df.iterrows():
    r = process_ticket(row["ticket_id"], row["customer_message"])
    results.append(r.model_dump())

results_df = pd.DataFrame(results)
results_df[["ticket_id", "category", "risk_level", "trust_score", "decision", "reason"]]


## 7. Run Evaluation

The evaluation harness compares Aegis predictions against expected labels.


In [ ]:
from src.evaluator import evaluate_sample

report = evaluate_sample("data/sample_tickets.csv")
report["metrics"]


## 8. Per-Ticket Evaluation Results


In [ ]:
eval_df = pd.DataFrame(report["rows"])
eval_df[[
    "ticket_id",
    "expected_category",
    "predicted_category",
    "category_match",
    "expected_risk",
    "predicted_risk",
    "risk_match",
    "expected_decision",
    "predicted_decision",
    "decision_match",
    "trust_score",
]]


## 9. Concept Summary

Aegis demonstrates:

### MCP-style tool use

The system retrieves policy text through:

policy_lookup(category)


### A2A-style communication

Agents pass structured messages:

TriageResult → DraftResult → JudgeResult → FinalResult


### Trust-gated support automation

The draft response is not final until the judge agent assigns a trust score and routing decision.


## 10. Current MVP Limitations

This notebook demonstrates the rule-based MVP.

Future improvements:

- Gemini-powered triage, drafting, and judging
- Real MCP server for policy lookup
- Larger evaluation dataset
- LLM-as-judge evaluation
- Web or CLI interface
